In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn.utils.rnn import pad_sequence


def create_sequences(df, group_cols=['user_id', 'skill_id'], block_size=512):
    """
    Groups df by group_cols and returns a list of sequences.
    """
    sequences = []
    group_keys = []

    for key, group in df.groupby(group_cols):
        seq = group[['correct', 'skill_id']].values  # (T, 2)

        if len(seq) > block_size:
            seq = seq[:block_size]
        sequences.append(torch.tensor(seq, dtype=torch.float32))
        group_keys.append(key)

    return sequences, group_keys


class BKTSequenceDataset(Dataset):
    def __init__(self, sequences, group_keys=None):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


def bkt_collate_fn(batch):
    obs = pad_sequence(batch, batch_first=True, padding_value=-1)
    return obs, obs


def get_data_loaders(
        df,
        group_cols=['user_id', 'skill_id'],
        block_size=512,
        batch_size=32):

    sequences, group_keys = create_sequences(df, group_cols=group_cols, block_size=block_size)

    np.random.seed(42)  # for reproducibility
    indices = np.arange(len(sequences))
    np.random.shuffle(indices)
    split = int(0.75 * len(indices))
    train_idx, val_idx = indices[:split], indices[split:]

    train_sequences = [sequences[i] for i in train_idx]
    val_sequences = [sequences[i] for i in val_idx]

    train_dataset = BKTSequenceDataset(train_sequences)
    val_dataset = BKTSequenceDataset(val_sequences)

    train_lengths = [len(seq) for seq in train_sequences]
    weights = torch.tensor([np.log(length + 1) for length in train_lengths], dtype=torch.float32)
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler,
        collate_fn=bkt_collate_fn,
        num_workers=0
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=bkt_collate_fn,
        num_workers=0
    )

    return train_loader, val_loader